In [8]:
pip install torch==2.8.0 torchaudio==2.8.0

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import tarfile
import random
import io
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import download_asset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import librosa
import whisper
from jiwer import wer
from tqdm.auto import tqdm
import IPython.display as ipd
from scipy.signal import correlate

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4 
    epochs = 10
    lr = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    n_fft = 640 
    win_len = 320
    hop_len = 160
    
    tar_path = "ru_train_0_19.tar"
    tsv_path = "train(1).tsv"
    extract_dir = "./ru_train_data"

In [4]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

if not os.path.exists(Config.extract_dir) and os.path.exists(Config.tar_path):
    print("Распаковка архива...")
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar:
        tar.extractall(path=Config.extract_dir)
    print("Распаковка завершена.")

if os.path.exists(Config.tsv_path):
    df_train = pd.read_csv(Config.tsv_path, sep='\t')
    reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}
else:
    reference_dict = {}
    print("ВНИМАНИЕ: Файл TSV не найден. Оценка WER не будет работать без текстов.")

def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

print("Загрузка ассетов шума...")
babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
BABBLE_WAVEFORM, sr_b = torchaudio.load(babble_path)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
RIR_WAVEFORM, sr_r = torchaudio.load(rir_path)
RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

Загрузка ассетов шума...


/tmp/ipykernel_9678/1716827913.py:25: UserWarning: torchaudio.utils.download.download_asset has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packa

In [5]:
def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)
        
    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

class SpeechEnhancementDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec
        self.files = []
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith('.mp3') and f in ref_dict:
                    self.files.append(os.path.join(root, f))

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        text = self.ref_dict[filename]

        waveform, sr = torchaudio.load(file_path)
        if sr != Config.sr:
            waveform = T.Resample(sr, Config.sr)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if self.is_train:
            max_samples = int(self.max_len_sec * Config.sr)
            if waveform.shape[-1] > max_samples:
                start = random.randint(0, waveform.shape[-1] - max_samples)
                waveform = waveform[:, start:start + max_samples]
            else:
                pad_len = max_samples - waveform.shape[-1]
                waveform = F.pad(waveform, (0, pad_len))
        
        clean = waveform
        noisy = apply_noise(clean) if self.is_train else clean

        return noisy.squeeze(0), clean.squeeze(0), text

def collate_fn(batch):
    noisy_list, clean_list, texts = [], [], []
    for noisy, clean, text in batch:
        noisy_list.append(noisy)
        clean_list.append(clean)
        texts.append(text)

    noisy_padded = pad_sequence(noisy_list, batch_first=True)
    clean_padded = pad_sequence(clean_list, batch_first=True)
    return noisy_padded, clean_padded, texts

In [6]:
class ComplexBatchNorm(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_r = nn.BatchNorm2d(num_features)
        self.bn_i = nn.BatchNorm2d(num_features)
    def forward(self, r, i):
        return self.bn_r(r), self.bn_i(i)

class ComplexConv2d(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride=(1,1), padding=(0,0)):
        super().__init__()
        self.conv_r = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.conv_i = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
    def forward(self, r, i):
        return self.conv_r(r) - self.conv_i(i), self.conv_r(i) + self.conv_i(r)

class ComplexConvTranspose2d(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride=(1,1), padding=(0,0), output_padding=(0,0)):
        super().__init__()
        self.conv_r = nn.ConvTranspose2d(in_c, out_c, kernel_size, stride, padding, output_padding, bias=False)
        self.conv_i = nn.ConvTranspose2d(in_c, out_c, kernel_size, stride, padding, output_padding, bias=False)
    def forward(self, r, i):
        return self.conv_r(r) - self.conv_i(i), self.conv_r(i) + self.conv_i(r)

class CFSMN(nn.Module):
    def __init__(self, channels, memory_size=20, axis='freq'):
        super().__init__()
        self.axis = axis
        self.memory_size = memory_size
        self.conv_r = nn.Conv1d(channels, channels, memory_size + 1, groups=channels, bias=False)
        self.conv_i = nn.Conv1d(channels, channels, memory_size + 1, groups=channels, bias=False)

    def forward(self, r, i):
        B, C, F_dim, T_dim = r.shape
        if self.axis == 'freq':
            r_in = r.permute(0, 3, 1, 2).reshape(B * T_dim, C, F_dim)
            i_in = i.permute(0, 3, 1, 2).reshape(B * T_dim, C, F_dim)
        else:
            r_in = r.permute(0, 2, 1, 3).reshape(B * F_dim, C, T_dim)
            i_in = i.permute(0, 2, 1, 3).reshape(B * F_dim, C, T_dim)

        r_pad = F.pad(r_in, (self.memory_size, 0))
        i_pad = F.pad(i_in, (self.memory_size, 0))

        out_r = self.conv_r(r_pad) - self.conv_i(i_pad)
        out_i = self.conv_r(i_pad) + self.conv_i(r_pad)

        res_r, res_i = r_in + out_r, i_in + out_i

        if self.axis == 'freq':
            return res_r.view(B, T_dim, C, F_dim).permute(0, 2, 3, 1), res_i.view(B, T_dim, C, F_dim).permute(0, 2, 3, 1)
        return res_r.view(B, F_dim, C, T_dim).permute(0, 2, 1, 3), res_i.view(B, F_dim, C, T_dim).permute(0, 2, 1, 3)

class CCBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        inter_channels = max(1, channels // 8)

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, inter_channels, 1),
            nn.ReLU(),
            nn.Conv2d(inter_channels, channels, 1),
            nn.Sigmoid()
        )
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, r, i):
        mag = torch.sqrt(r**2 + i**2 + 1e-8)
        ca = self.channel_attn(mag)
        r, i, mag = r * ca, i * ca, mag * ca
        avg_m = torch.mean(mag, dim=1, keepdim=True)
        max_m = torch.max(mag, dim=1, keepdim=True)[0]
        sa = self.spatial_attn(torch.cat([avg_m, max_m], dim=1))
        return r * sa, i * sa

In [7]:
class FRCRN(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_fft, self.hop, self.win = Config.n_fft, Config.hop_len, Config.win_len
        self.register_buffer('window', torch.hann_window(self.win))

        self.encoders = nn.ModuleList([
            self._make_enc_block(1 if k==0 else 128) for k in range(6)
        ])

        self.bottleneck = nn.Sequential(
            CFSMN(128, 20, 'time'),
            CFSMN(128, 20, 'time')
        )

        self.decoders = nn.ModuleList([
            self._make_dec_block(
                in_c=256,
                out_c=(128 if k < 5 else 1),
                out_pad=(0, 0),
                attn_c=128
            ) for k in range(6)
        ])

        self.alpha = nn.Parameter(torch.tensor([0.95]))

    def _make_enc_block(self, in_c):
        return nn.ModuleDict({
            'conv': ComplexConv2d(in_c, 128, (5, 2), stride=(2, 1), padding=(2, 0)),
            'bn': ComplexBatchNorm(128),
            'fsmn': CFSMN(128, 20, 'freq')
        })

    def _make_dec_block(self, in_c, out_c, out_pad, attn_c):
        return nn.ModuleDict({
            'attn': CCBAM(attn_c),
            'conv': ComplexConvTranspose2d(in_c, out_c, (5, 2), stride=(2, 1), padding=(2, 0), output_padding=out_pad),
            'bn': ComplexBatchNorm(out_c),
            'fsmn': CFSMN(out_c, 20, 'freq')
        })

    def forward(self, x):
        B, _, L = x.shape
        stft = torch.stft(x.squeeze(1), self.n_fft, self.hop, self.win, self.window,
                          return_complex=True, center=True)
        noisy_r, noisy_i = stft.real.unsqueeze(1), stft.imag.unsqueeze(1)

        r, i = noisy_r, noisy_i
        skips = []

        for enc in self.encoders:
            r = F.pad(r, (1, 0, 0, 0))
            i = F.pad(i, (1, 0, 0, 0))

            r, i = enc['conv'](r, i)
            r, i = r[:, :, :, :-1], i[:, :, :, :-1]

            r, i = enc['bn'](r, i)
            r, i = F.leaky_relu(r, 0.2), F.leaky_relu(i, 0.2)
            r, i = enc['fsmn'](r, i)

            skips.append((r, i))

        r, i = self.bottleneck[0](r, i)
        r, i = self.bottleneck[1](r, i)

        for idx, dec in enumerate(self.decoders):
            skip_r, skip_i = skips[-(idx+1)]
            sr, si = dec['attn'](skip_r, skip_i)

            if r.shape[2:] != sr.shape[2:]:
                r = F.interpolate(r, size=sr.shape[2:], mode='bilinear', align_corners=True)
                i = F.interpolate(i, size=si.shape[2:], mode='bilinear', align_corners=True)

            r, i = torch.cat([r, sr], 1), torch.cat([i, si], 1)

            r = F.pad(r, (1, 0, 0, 0))
            i = F.pad(i, (1, 0, 0, 0))
            r, i = dec['conv'](r, i)
            r, i = r[:, :, :, :-1], i[:, :, :, :-1]

            r, i = dec['bn'](r, i)
            if idx < 5:
                r, i = F.leaky_relu(r, 0.2), F.leaky_relu(i, 0.2)
                r, i = dec['fsmn'](r, i)

        m_r, m_i = torch.tanh(r), torch.tanh(i)
        est_r = noisy_r * m_r - noisy_i * m_i
        est_i = noisy_r * m_i + noisy_i * m_r

        est_wav = torch.istft(torch.complex(est_r.squeeze(1), est_i.squeeze(1)),
                              self.n_fft, self.hop, self.win, self.window,
                              length=L, center=True).unsqueeze(1)

        return self.alpha * est_wav + (1 - self.alpha) * x

In [8]:
class STFTLoss(nn.Module):
    def __init__(self, n_fft, hop_length, win_length):
        super(STFTLoss, self).__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.window = nn.Parameter(torch.hann_window(win_length), requires_grad=False)

    def forward(self, x, y):
        x_stft = torch.stft(x, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)
        y_stft = torch.stft(y, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)

        x_mag = torch.abs(x_stft) + 1e-8
        y_mag = torch.abs(y_stft) + 1e-8

        sc_loss = torch.linalg.norm(y_mag - x_mag, ord='fro', dim=(1, 2)) / (torch.linalg.norm(y_mag, ord='fro', dim=(1, 2)) + 1e-8)
        sc_loss = sc_loss.mean()
        
        mag_loss = F.l1_loss(torch.log(y_mag), torch.log(x_mag))

        return sc_loss + mag_loss

class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[120, 240, 480], win_lengths=[600, 1200, 2400]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.loss_layers = nn.ModuleList([
            STFTLoss(fs, hs, wl) for fs, hs, wl in zip(fft_sizes, hop_sizes, win_lengths)
        ])

    def forward(self, x, y):
        mr_stft_loss = 0
        for layer in self.loss_layers:
            mr_stft_loss += layer(x, y)
        return mr_stft_loss / len(self.loss_layers)

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mr_stft = MultiResolutionSTFTLoss(
            fft_sizes=[512, 1024, 2048],
            hop_sizes=[128, 256, 512],
            win_lengths=[512, 1024, 2048]
        )

    def forward(self, denoised, clean):
        target = clean - torch.mean(clean, dim=-1, keepdim=True)
        estimated = denoised - torch.mean(denoised, dim=-1, keepdim=True)

        dot = torch.sum(target * estimated, dim=-1, keepdim=True)
        norm = torch.sum(target ** 2, dim=-1, keepdim=True) + 1e-8

        target_scaled = (dot / norm) * target
        noise = estimated - target_scaled

        si_sdr_loss = -10 * torch.log10(
            (torch.sum(target_scaled ** 2, dim=-1) + 1e-8) /
            (torch.sum(noise ** 2, dim=-1) + 1e-8)
        )
        si_sdr_loss = si_sdr_loss.mean()
        stft_loss = self.mr_stft(denoised, clean)

        return (1 - self.alpha) * si_sdr_loss + self.alpha * stft_loss

In [9]:
def np_si_sdr(ref, est):
    ref, est = ref.reshape(-1), est.reshape(-1)
    dot = np.dot(ref, est)
    norm = np.linalg.norm(ref)**2
    proj = (dot / (norm + 1e-8)) * ref
    noise = est - proj
    return 10 * np.log10((np.linalg.norm(proj)**2) / (np.linalg.norm(noise)**2 + 1e-8) + 1e-8)

def evaluate_and_listen(model, device, val_dataset, limit=10):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()

    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": [], "sisdr_n": [], "sisdr_d": [], "samples": []} for n in noise_types}
    clean_wer = []

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_np = clean_wav.numpy()

            t_clean = asr.transcribe(clean_np, fp16=False, language='ru')['text']
            clean_wer.append(wer(ref_text, clean_text(t_clean)))

            clean_tensor_cpu = clean_wav.unsqueeze(0)

            for n_type in noise_types:
                
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)
                noisy_in = noisy_tensor.unsqueeze(1) 
                
                denoised_tensor = model(noisy_in)
                denoised_tensor = denoised_tensor.squeeze(1).squeeze(0)

                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.cpu().numpy()

                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))
                stats[n_type]["sisdr_n"].append(np_si_sdr(clean_np, noisy_np))
                stats[n_type]["sisdr_d"].append(np_si_sdr(clean_np, denoised_np))

                if len(stats[n_type]["samples"]) < 3:
                    stats[n_type]["samples"].append({
                        "clean": clean_np, "noisy": noisy_np, "denoised": denoised_np,
                        "text": ref_text, "t_d": t_d
                    })

    print(f"\nBaseline WER (Clean Audio): {np.mean(clean_wer):.4f}")
    header = f"{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denois':<10} | {'Gain WER':<10} | {'SI-SDR Gain':<10}"
    print(header)

    for n_type in noise_types:
        w_n = np.mean(stats[n_type]["wer_n"])
        w_d = np.mean(stats[n_type]["wer_d"])
        s_n = np.mean(stats[n_type]["sisdr_n"])
        s_d = np.mean(stats[n_type]["sisdr_d"])
        print(f"{n_type:<10} | {w_n:<10.4f} | {w_d:<10.4f} | {w_n - w_d:<10.4f} | {s_d - s_n:<10.2f} dB")

    print("\nАУДИО ПРИМЕРЫ (по 3 на каждый шум):")
    for n_type in noise_types:
        print(f"ТИП ШУМА: {n_type.upper()}")
        for i, samp in enumerate(stats[n_type]["samples"]):
            print(f"\nПример {i+1}")
            print(f"Оригинальный текст: {samp['text']}")
            print(f"Распознано (Denoised): {samp['t_d']}")

            print("1. Чистое аудио:")
            ipd.display(ipd.Audio(samp["clean"], rate=Config.sr))

            print("2. Зашумленное аудио:")
            ipd.display(ipd.Audio(samp["noisy"], rate=Config.sr))

            print("3. Очищенное аудио:")
            ipd.display(ipd.Audio(samp["denoised"], rate=Config.sr))

In [10]:
dataset = SpeechEnhancementDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)

model = FRCRN().to(Config.device)
criterion = CombinedLoss(alpha=0.8).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr, fused=True)

In [ ]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{Config.epochs}")
    
    for noisy, clean, _ in pbar:
        noisy = noisy.unsqueeze(1).to(Config.device)
        clean = clean.to(Config.device)

        optimizer.zero_grad()

        denoised = model(noisy)
        denoised = denoised.squeeze(1)

        loss = criterion(denoised, clean)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch} | Avg Loss: {total_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), 'frcrn_weights_final.pth')

evaluate_and_listen(model, Config.device, val_dataset, limit=10)

Epoch 1/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 1 | Avg Loss: 0.4644


Epoch 2/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 2 | Avg Loss: -0.1278


Epoch 3/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 3 | Avg Loss: -0.2060


Epoch 4/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 4 | Avg Loss: -0.3365


Epoch 5/10:   0%|          | 0/5954 [00:00<?, ?it/s]

In [21]:
torch.save(model.state_dict(), '5_epoh_frcrn.pth')

In [ ]:
for epoch in range(6, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{Config.epochs}")
    
    for noisy, clean, _ in pbar:
        noisy = noisy.unsqueeze(1).to(Config.device)
        clean = clean.to(Config.device)

        optimizer.zero_grad()

        denoised = model(noisy)
        denoised = denoised.squeeze(1)

        loss = criterion(denoised, clean)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch} | Avg Loss: {total_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), 'frcrn_weights_final.pth')

evaluate_and_listen(model, Config.device, val_dataset, limit=10)

Epoch 6/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 6 | Avg Loss: -0.4406


Epoch 7/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 7 | Avg Loss: -0.4700


Epoch 8/10:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 8 | Avg Loss: -0.4995


Epoch 9/10:   0%|          | 0/5954 [00:00<?, ?it/s]

In [23]:
torch.save(model.state_dict(), '8_epoh_frcrn.pth')

In [11]:
dataset = SpeechEnhancementDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True)

model = FRCRN().to(Config.device)
criterion = CombinedLoss(alpha=0.8).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr, fused=True)

In [12]:
model.load_state_dict(torch.load('8_epoh_frcrn.pth'))

<All keys matched successfully>

In [13]:
for epoch in range(9, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{Config.epochs}")
    
    for noisy, clean, _ in pbar:
        noisy = noisy.unsqueeze(1).to(Config.device)
        clean = clean.to(Config.device)

        optimizer.zero_grad()

        denoised = model(noisy)
        denoised = denoised.squeeze(1)

        loss = criterion(denoised, clean)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch} | Avg Loss: {total_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), 'frcrn_weights_final.pth')

evaluate_and_listen(model, Config.device, val_dataset, limit=10)

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Epoch 9/10:   0%|          | 0/5954 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

KeyboardInterrupt: 

In [11]:
state_dict = torch.load('8_epoh_frcrn.pth', map_location=Config.device)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [18]:
seed_everything(42)
evaluate_and_listen(model, Config.device, val_dataset, limit=20)

Evaluation:   0%|          | 0/20 [00:00<?, ?it/s]


Baseline WER (Clean Audio): 0.4156
Noise Type | WER Noisy  | WER Denois | Gain WER   | SI-SDR Gain
babble     | 0.5376     | 0.5010     | 0.0366     | 5.58       dB
rir        | 1.0000     | 1.0000     | 0.0000     | 26.80      dB
white      | 0.5670     | 0.7300     | -0.1630    | 8.25       dB

АУДИО ПРИМЕРЫ (по 3 на каждый шум):
ТИП ШУМА: BABBLE

Пример 1
Оригинальный текст: белым быком возрос над землей муууу
Распознано (Denoised):  возрос над землей.
1. Чистое аудио:


2. Зашумленное аудио:


3. Очищенное аудио:



Пример 2
Оригинальный текст: тем не менее он должен и будет идти вперед
Распознано (Denoised):  Он должен идти в детектив перед.
1. Чистое аудио:


...

2. Зашумленное аудио:


3. Очищенное аудио:



Пример 3
Оригинальный текст: конференция также должна делать свою работу в частности рассматривать свою нынешнюю повестку дня
Распознано (Denoised):  если рассматривать свою нынешнюю повестку дня.
1. Чистое аудио:


2. Зашумленное аудио:


3. Очищенное аудио:


In [13]:
from jiwer import process_words
import torch
import numpy as np
from tqdm.auto import tqdm
import whisper

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0) # [1, L]
            
            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)

                noisy_in = noisy_tensor.unsqueeze(1) 
                denoised_tensor = model(noisy_in)
                
                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()
                
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
                
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
                
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    for k in ["wer_n", "s_n", "d_n", "i_n"]: stats[n_type][k].append(0.0)
                
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    for k in ["wer_d", "s_d", "d_d", "i_d"]: stats[n_type][k].append(0.0)

    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print("\n" + header)
    print("-" * 75)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
    
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
    
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
    
        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_and_listen_components(model, Config.device, val_dataset, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]


Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
---------------------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.5010 (0.1244/0.3720/0.0045) | 0.0366  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0000 (0.1740/0.8260/0.0000) | 0.0000  
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.7300 (0.2361/0.4653/0.0286) | -0.1630 
